In [ ]:
import os
from langgraph.graph import StateGraph, START, END
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import TypedDict, Annotated, Sequence
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, SystemMessage, ToolMessage, HumanMessage
from operator import add as add_messages
from langchain_core.tools import tool
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq
import time
from langgraph.checkpoint.memory import MemorySaver
import uuid
import gradio as gr

In [3]:
load_dotenv()

api_embedd = os.getenv("api")
api_llm = os.getenv("api_proq")

In [ ]:
class stateAgent(TypedDict):
    messages: Annotated[Sequence[BaseMessage],add_messages]

In [ ]:
pdf_path= r"C:\Users\ASUS\Hoc_DL\learning-DL\RAG_pdf\data\raw\LuanNgu.pdf"
pdf_load= PyMuPDFLoader(pdf_path)
pdf = pdf_load.load()

In [ ]:
embedd = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    api_key=api_embedd
)

model= ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    temperature=0,
    api_key= api_llm
)

text_split= RecursiveCharacterTextSplitter(
    chunk_size= 1000,
    chunk_overlap= 200
)

docs= text_split.split_documents(pdf)
print(len(docs))

408


In [ ]:
db= Chroma(
    embedding_function=embedd,
    collection_name= "vectorDB",
    persist_directory= r"C:\Users\ASUS\Hoc_DL\learning-DL\RAG_pdf\data\vectorstore"
)

In [ ]:
batch_size = 25
total = len(docs)

for i in range(0, total, batch_size):
    batch = docs[i : i + batch_size]
   
    db.add_documents(batch)
    
    print(f"{i + len(batch)}/{total}")
    
    if i + batch_size < total:
        time.sleep(45)

25/408
50/408
75/408
100/408
125/408
150/408
175/408
200/408
225/408
250/408
275/408
300/408
325/408
350/408
375/408
400/408
408/408


In [ ]:
retriever = db.as_retriever(search_kwargs={"k": 3})

In [ ]:
@tool
def retrieval_tool(query: str):
    """Sử dụng retriever_tool nếu câu hỏi liên quan đến nội dung tài liệu. 
    Nếu câu hỏi nằm ngoài phạm vi tài liệu hoặc là lời chào hỏi xã giao, hãy trả lời trực tiếp rằng bạn không có thông tin."""
    
    docs= retriever.invoke(query)
    
    if docs is None:
        return "Not found!!!"
    
    list_docs=[]
    for i, doc in enumerate(docs):
        list_docs.append(f"Docs {i+1}: \n{doc.page_content}")
    
    return "\n\n".join(list_docs)


@tool
def 

tools = [retrieval_tool]

llm_with_tools= model.bind_tools(tools)
    


In [ ]:
system_prompt = """
Bạn là một AI chuyên phân tích về cuốn Luận Ngữ của Khổng Tử.
Sử dụng retrieval_tool khi có bất kì câu hỏi nào liên quan đến cuốn sách này.
Hãy trả lời tường minh, bằng tiếng Việt và nếu cần thì có thể trích xuất từ tài liệu ra để làm bằng chứng.
Chỉ được trả lời bằng nội dung pdf được cung cấp.
Tài liệu, sách, nội dung, ... là các từ mà người ta thường mô tả về nội dung của pdf.
"""

def call_llm(state: stateAgent):
    messages = list(state['messages'])
    
    messages = [SystemMessage(content=system_prompt)] + messages
    
    rs= llm_with_tools.invoke(messages)
    
    return {"messages": [rs]}

In [ ]:
def action(state: stateAgent):
    
    tool_calls = state['messages'][-1].tool_calls
    results=[]
    
    for t in tool_calls:
        
        if t['name'] == 'retrieval_tool':
            # # t['args'] chứa câu hỏi đã được chuyển đổi thông qua quá trình suy luận
            result = retrieval_tool.invoke(t['args'].get('query', ''))
        else:
            result = "Tool not found!!!"
            
        results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        
    return {"messages": results}
        
        
            

In [ ]:
def should_cont(state: stateAgent):
    result = state['messages'][-1]
    
    if hasattr(result, 'tool_calls') and len(result.tool_calls) > 0:
        return "retriever_agent"
    return 'end'

In [ ]:
graph = StateGraph(stateAgent)

graph.add_node('call_llm',call_llm)
graph.add_node('action',action)

graph.add_conditional_edges(
    "call_llm",
    should_cont,
    {'retriever_agent': "action", 'end': END}
)

graph.add_edge('action','call_llm')
graph.add_edge(START, "call_llm")

memory = MemorySaver()
rag= graph.compile(checkpointer=memory)


random_uuid = uuid.uuid4()
config = {"configurable": {"thread_id": str(random_uuid)}}

while True:
    ip =input("Your:") 
    if ip == '':
        break
    
    rs = rag.invoke(
        {'messages':[HumanMessage(content=ip)]},
        config= config
        )

    print("LLM:", rs['messages'][-1].content)

LLM: Tôi là một AI chuyên phân tích về cuốn Luận Ngữ của Khổng Tử. Tôi có thể giúp bạn trả lời các câu hỏi liên quan đến cuốn sách này. Nếu bạn có bất kỳ câu hỏi nào, hãy hỏi tôi!
LLM: Tôi không có thông tin cụ thể về "cái gì đây" trong ngữ cảnh của cuốn Luận Ngữ. Tuy nhiên, dựa trên nội dung của tài liệu, có thể thấy rằng nó liên quan đến các vấn đề về Nho học, văn hóa phương Đông, và các bài học rút từ Luận ngữ. Nếu bạn có thể cung cấp thêm thông tin hoặc câu hỏi cụ thể, tôi sẽ cố gắng giúp bạn tìm kiếm thông tin liên quan.
LLM: Cuốn Luận Ngữ là một trong những giá trị quý báu và độc đáo của Nho học, là tài sản chung của các nền văn hóa khu vực đồng văn (Trung Quốc, Đài Loan, Triều Tiên, Hàn Quốc, Nhật Bản và Việt Nam). Nó là sách do học trò và hậu thế ghi chép lại những lời nói, hành vi của Khổng Tử, học trò ông và người đương thời. Cuốn sách gồm 20 thiên (tương đương với 20 chương), mỗi thiên gồm nhiều bài, và được coi là một trong những tác phẩm tiêu biểu của Nho học.
LLM: Cuốn Lu

In [ ]:

def chatbot_response(message, history):
    # message: Câu hỏi mới nhất từ giao diện gõ vào
    # history: Lịch sử do UI tự lưu (ở đây ta bỏ qua vì LangGraph đã có MemorySaver)
    
    result = rag.invoke(
        {"messages": [HumanMessage(content=message)]},
        config=config
    )
    
    # Trả về đúng chuỗi str để giao diện in ra
    return result['messages'][-1].content

demo = gr.ChatInterface(
    fn=chatbot_response, 
    title="Hệ thống Agentic RAG",
    description="Luận Ngữ"
)

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


### Tiền xử lý cho Hán - Việt


In [4]:
import pandas as pd

In [5]:
df = pd.read_csv(r"C:\Users\ASUS\Hoc_DL\learning-DL\RAG_pdf\data\raw\vi_dictionary.csv")
df

,word,part_of_speech,meaning,example
0,a di đà phật,danh từ,Phật A Di Đà (vị Phật lớn nhất ở Cõi Cực Lạc; ...,na mô A Di Đà Phật!
1,a dua,động từ,"làm theo, bắt chước theo việc làm sai trái của...",a dua theo bọn xấu làm bậy
2,ả,danh từ,"(cũ) người con gái: ""Đầu lòng hai ả tố nga, Th...","""Đầu lòng hai ả tố nga, Thuý Kiều là chị em là..."
3,ả,danh từ,(khẩu ngữ) từ dùng để chỉ người phụ nữ nào đó ...,mấy ả gái điếm * ả đã thực hiện nhiều vụ lừa đảo
4,ả,danh từ,"(phương ngữ) chị: tại anh tại ả, tại cả đôi bê...","tại anh tại ả, tại cả đôi bên (tng)"
...,...,...,...,...
36759,mệt xác,NaN,"(khẩu ngữ) mệt, tốn công sức một cách vô ích, ...",nghĩ làm gì cho mệt xác!
36760,mệt nhọc,tính từ,mệt vì phải bỏ nhiều sức lực (nói khái quát): ...,một ngày lao động mệt nhọc
36761,mếu máo,động từ,từ gợi tả dáng miệng bị méo xệch đi khi đang k...,miệng mếu máo chực khóc
36762,mếu,động từ,(miệng) méo đi chực khóc: cười như mếu * khóc ...,cười như mếu * khóc dở mếu dở (tng)


In [ ]:
import json
import csv
from collections import defaultdict

dic = defaultdict(list)
csv_path = r"C:\Users\ASUS\Hoc_DL\learning-DL\RAG_pdf\data\raw\vi_dictionary.csv"
json_path = r"C:\Users\ASUS\Hoc_DL\learning-DL\RAG_pdf\data\raw\HanViet.json"

# Sử dụng csv.DictReader để đọc file
with open(csv_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    
    # Lúc này 'row' thực sự là một Dictionary
    for row in reader:
        word = row['word'].strip().lower()
        mean = row['meaning'].strip()
        ex = row['example'].strip()
        
        dic[word].append({
            'Nghĩa': mean,
            'Ví dụ': ex
        })

# Ghi ra file JSON
with open(json_path, mode='w', encoding='utf-8') as w:
    json.dump(dic, w, ensure_ascii=False, indent=2)

### Tiền xử lý cho pdf

In [6]:
loadpdf= PyMuPDFLoader(r"C:\Users\ASUS\Hoc_DL\learning-DL\RAG_pdf\data\raw\LuanNgu.pdf")
D = loadpdf.load()
D

[Document(metadata={'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2012-01-12T10:51:22-08:00', 'source': 'C:\\Users\\ASUS\\Hoc_DL\\learning-DL\\RAG_pdf\\data\\raw\\LuanNgu.pdf', 'file_path': 'C:\\Users\\ASUS\\Hoc_DL\\learning-DL\\RAG_pdf\\data\\raw\\LuanNgu.pdf', 'total_pages': 179, 'format': 'PDF 1.5', 'title': '', 'author': 'D Le', 'subject': '', 'keywords': '', 'moddate': '2012-01-12T10:51:22-08:00', 'trapped': '', 'modDate': "D:20120112105122-08'00'", 'creationDate': "D:20120112105122-08'00'", 'page': 0}, page_content='1 \nLuận Ngữ - Khổng Tử --  Phùng Hoài Ngọc  biên giả  ---  www.vietnamvanhien.net \n \nLuận Ngữ \nKhổng Tử \n论语 \nPhùng Hoài Ngọc \nbiên dịch – chủ giải - bàn luận \nĐại học An Giang, 2011 \n \n \nKhổng tử dạy học'),
 Document(metadata={'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2012-01-12T10:51:22-08:00', 'source': 'C:\\Users\\ASUS\\Hoc_DL\\learning-DL\\RAG_pdf\\data\\raw\\LuanNgu

## Đầu tiên chuyển nó sang json trước và ở đây chỉ lưu gồm page and content

##### Test

In [7]:
D[11]

Document(metadata={'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2012-01-12T10:51:22-08:00', 'source': 'C:\\Users\\ASUS\\Hoc_DL\\learning-DL\\RAG_pdf\\data\\raw\\LuanNgu.pdf', 'file_path': 'C:\\Users\\ASUS\\Hoc_DL\\learning-DL\\RAG_pdf\\data\\raw\\LuanNgu.pdf', 'total_pages': 179, 'format': 'PDF 1.5', 'title': '', 'author': 'D Le', 'subject': '', 'keywords': '', 'moddate': '2012-01-12T10:51:22-08:00', 'trapped': '', 'modDate': "D:20120112105122-08'00'", 'creationDate': "D:20120112105122-08'00'", 'page': 11}, page_content='12 \nLuận Ngữ - Khổng Tử --  Phùng Hoài Ngọc  biên giả  ---  www.vietnamvanhien.net \n \n1.9  \n曾子曰：慎终追远，民德归厚矣 。  \nTăng tử viết: Thận chung truy viễn, dân đức quy hậu hĩ. \nTăng tử nói: Cẩn thận lo tang lễ cha mẹ, thƣờng tƣởng nhớ tổ tiên xƣa, dân chúng cảm đức \nmà theo về. \n \n(Lời bàn: “truy viễn” (nhớ ngƣời xƣa) rất đa nghĩa. Tƣởng nhớ ông bà tổ tiên, nhớ họ hàng nơi \nxa, nhớ lời dạy của bậc thánh nhân thời trƣớc… Đây l

In [8]:
import re
def clean_pdf_header_footer(text: str) -> str:
    """
    Hàm xóa bỏ phần header/footer chứa số trang và thông tin xuất bản
    lặp lại trong file PDF Luận Ngữ.
    """
    # Định nghĩa mẫu Regex
    pattern = r"\d+\s*\n\s*Luận Ngữ - Khổng Tử --\s*Phùng Hoài Ngọc\s*biên giả\s*---\s*www\.vietnamvanhien\.net\s"
    
    # Thực hiện thay thế chuỗi khớp với mẫu bằng chuỗi rỗng ("")
    cleaned_text = re.sub(pattern, "", text)
    
    return cleaned_text

In [ ]:
# D_list=[]
# for i, doc in enumerate(D):
#     doc = clean_pdf_header_footer(doc.page_content)
#     doc =doc.replace('Ƣ','Ư')
#     doc = doc.replace('ƣ','ư')
#     D_list.append(doc)
    
# df = pd.DataFrame(D_list)
    
    

In [ ]:
# df.to_excel('test.xlsx')

In [9]:
tx=''
for i, doc in enumerate(D):
    doc = clean_pdf_header_footer(doc.page_content)
    doc =doc.replace('Ƣ','Ư')
    doc = doc.replace('ƣ','ư')
    tx+=doc


# import re
# tx = re.sub(r'\n(?!\s)', '\n ', tx)

In [10]:
tx

'\n \nLuận Ngữ \nKhổng Tử \n论语 \nPhùng Hoài Ngọc \nbiên dịch – chủ giải - bàn luận \nĐại học An Giang, 2011 \n \n \nKhổng tử dạy học\n \n  \n  MỤC LỤC  \nLời nói đầu. 2 \n1.     学而 Học nhi 8 \n2.     为政Vi chính. 13 \n3.     八佾 Bát dật 20 \n4.     里仁 Lý nhân. 29 \n5.     公冶长Công Dã Tràng. 35 \n6.     雍也Ung dã. 44 \n7.     述而 Thuật nhi 53 \n8.     泰伯 Thái Bá. 63 \n9.     子罕Tử hãn. 69 \n10.       言乡党Hương đảng. 78 \n11.       先进Tiên tiến. 85 \n12.       颜渊 Nhan Uyên. 95 \n13.       子路Tử Lộ. 103 \n14.       宪问Hiến vấn. 113 \n15.       卫灵公Vệ Linh công. 126 \n16.       季氏 Quí thị 137 \n17.       阳货  Dương Hóa. 143 \n18.       微子 Vi Tử.. 152 \n19.       子张 Tử Trương. 157 \n20.       尧曰 Nghiêu viết 165\n \nBÀI TẬP NGHIÊN CỨU LUẬN NGỮ.. 167 \nPHỤ LỤC- SƠ LƯỢC LỊCH SỬ TRUNG QUỐC………………………..…………..168 \nTÀI LIỆU THAM KHẢO ……………………………………………………………..169 \n  \n  \nLỜI NÓI ĐẦU \nVăn học Trung Quốc thời cổ đại còn gọi Văn học tiên Tần, có 4 thành tựu chính: \n1/Thần thoại, \n2/ Ca dao dân ca (Kinh Thi) \

In [ ]:
with open('test.txt', mode='w', encoding='utf-8') as f:
    f.write(tx)

In [ ]:
parts = re.split(r'Hết\s+thiên\s+\d+', tx)

In [ ]:
parts[0]

'\n \nLuận Ngữ \nKhổng Tử \n论语 \nPhùng Hoài Ngọc \nbiên dịch – chủ giải - bàn luận \nĐại học An Giang, 2011 \n \n \nKhổng tử dạy học\n \n  \n  MỤC LỤC  \nLời nói đầu. 2 \n1.     学而 Học nhi 8 \n2.     为政Vi chính. 13 \n3.     八佾 Bát dật 20 \n4.     里仁 Lý nhân. 29 \n5.     公冶长Công Dã Tràng. 35 \n6.     雍也Ung dã. 44 \n7.     述而 Thuật nhi 53 \n8.     泰伯 Thái Bá. 63 \n9.     子罕Tử hãn. 69 \n10.       言乡党Hương đảng. 78 \n11.       先进Tiên tiến. 85 \n12.       颜渊 Nhan Uyên. 95 \n13.       子路Tử Lộ. 103 \n14.       宪问Hiến vấn. 113 \n15.       卫灵公Vệ Linh công. 126 \n16.       季氏 Quí thị 137 \n17.       阳货  Dương Hóa. 143 \n18.       微子 Vi Tử.. 152 \n19.       子张 Tử Trương. 157 \n20.       尧曰 Nghiêu viết 165\n \nBÀI TẬP NGHIÊN CỨU LUẬN NGỮ.. 167 \nPHỤ LỤC- SƠ LƯỢC LỊCH SỬ TRUNG QUỐC………………………..…………..168 \nTÀI LIỆU THAM KHẢO ……………………………………………………………..169 \n  \n  \nLỜI NÓI ĐẦU \nVăn học Trung Quốc thời cổ đại còn gọi Văn học tiên Tần, có 4 thành tựu chính: \n1/Thần thoại, \n2/ Ca dao dân ca (Kinh Thi) \

In [ ]:
len(parts)

20

In [ ]:
for i,part in enumerate(parts):
    with open(f'test{i}.txt', mode='w', encoding='utf-8') as f:
        f.write(part)

In [ ]:
with open(r'C:\Users\ASUS\Hoc_DL\learning-DL\RAG_pdf\pipeline\test19.txt', mode='r', encoding='utf-8') as f:
    thien = f.read()
    thien20 = re.split(r'Hết\s+', thien)
    j = 0
    for i in range(19,21):
        with open(f'test{i}.txt', mode='w', encoding='utf-8') as ff:
            ff.write(thien20[j])
        j+=1


In [ ]:
with open(r'C:\Users\ASUS\Hoc_DL\learning-DL\RAG_pdf\pipeline\test0.txt', mode='r', encoding='utf-8') as f:
    thien = f.read()
    thien20 = re.split(r'Biên giả', thien)
    j = 0
    for i in range(-1,1):
        with open(f'test{i}.txt', mode='w', encoding='utf-8') as ff:
            ff.write(thien20[j])
        j+=1
    

In [20]:
from collections import defaultdict
list_thien = [
 'Học nhi',
 'Vi chính',
 'Bát dật',
 'Lý nhân',
 'Công Dã Tràng',
 'Ung dã',
 'Thuật nhi',
 'Thái Bá',
 'Tử hãn',
 'Hương đảng',
 'Tiên tiến',
 'Nhan Uyên',
 'Tử Lộ',
 'Hiến vấn',
 'Vệ Linh công',
 'Quí thị',
 'Dương Hóa',
 'Vi Tử',
 'Tử Trương',
 'Nghiêu viết'
]
list_thien_han=['学而','为政','八佾','里仁','公冶长','雍也','述而','泰伯','子罕','言乡党','先进','颜渊','子路','宪问','卫灵公','季氏','阳货','微子','子张','尧曰']


dicc = defaultdict(dict)

for i in range(20):
    dicc[f'Thiên {i+1}']['Tên thiên']= list_thien[i]
    dicc[f'Thiên {i+1}']['Chữ Hán']= list_thien_han[i]
    


        

In [21]:
#9.2
Thien_du= [2,4,5,6,10,11,12,13,14,15,16,17,18,19,20]

In [22]:
total =0
for i in range(0,20):
    with open(f'C:/Users/ASUS/Hoc_DL/learning-DL/RAG_pdf/pipeline/test{i}.txt', mode='r', encoding='utf-8') as f:
        thien= f.read()
        thien = thien.replace('．', '.')
        thien = thien.replace('·', '.')
        thien = thien.replace('\xa0', ' ')
        thien = thien.replace('\u3000', ' ')
        pattern = r'(?m)^\d+\.\d+.*?(?=^\d+\.\d+|\Z)'

        matches = re.split(r'\d+\.\s*\d+', thien)
        
        # print(len(matches))
        # if len(matches) <= sobai[i]:
        #     print(i, len(matches), sobai[i])
        k = 1
        for j in range(1,len(matches)):
            
            new=matches[j].replace('·','.')
            # new= new.replace('.','.')
            if len(new)>15:
                
                if i+1 == 9:
                    if j==2:
                        continue
                if i+1 in Thien_du:
                    if j==1:
                        continue
                
                dicc[f'Thiên {i+1}'][f'Bài {i+1}.{k}']= new
                k+=1
            #     total+=1
            #     with open(f'C:/Users/ASUS/Hoc_DL/learning-DL/RAG_pdf/pipeline/thien/test{i}_{j}.txt', mode='w', encoding='utf-8') as ff:
            #         ff.write(new)
            # else:
            #     print(f"{i}_{j}")
                # print(new)


In [23]:
total-20

-20

In [186]:
sobai=[16,24,26,26,28,30,38,21,31,27,25,24,30,44,42,14,25,11,25,5]

In [24]:
dicc

defaultdict(dict,
            {'Thiên 1': {'Tên thiên': 'Học nhi',
              'Chữ Hán': '学而',
              'Bài 1.1': '\n \n子曰: 学而时习之,不亦悅乎？ \n有朊自远方来,不亦乐乎? \n人不知 而不愠,不亦君子乎 ? \nTử viết: Học nhi thời tập chi, bất diệc duyệt hồ ? \nHữu bằng tự viễn phương lai, bất diệc lạc hồ ? \nNhân bất tri nhi bất uấn, bất diệc quân tử hồ ? \nKhổng tử nói: Học thì phải luyện tập, chẳng vui lắm sao ? \nCó bạn hữu nơi xa đến thăm, chẳng mừng lắm sao? \nNgười chẳng hiểu ta mà ta không buồn giận họ, thế chẳng phải người quân tử ư ? \n  \n(Lời bàn: Bài học đầu tiên, Khổng tử nói về niềm vui “học và hành”, niềm vui đón “bạn phương \nxa” và…nhắc đừng buồn khi có người hiểu lầm ta)  \n',
              'Bài 1.2': '  \n有子曰:“兲为人也孝悌, 而好犯上者,鲜矣; 不好犯上,而好作乱者,未之有也。君子务本,本立\n而道生。孝悌也者, 兲为仁之本与？” \nHữu tử viết: Kỳ vi nhân dã hiếu đễ, nhi hiếu phạm thượng giả, tiển hĩ; bất hiếu phạm thượng, nhi \nhiếu tác loạn giả, vị chi hữu dã. Quân tử vụ bản, bản lập nhi đạo sinh. Hiếu đễ dã giả, kỳ vi nhân \nchi bản dữ !\n \nHữu tử n

In [30]:
with open('luanngu.json', mode='w', encoding='utf-8') as w:
    json.dump(dicc, w, ensure_ascii=False, indent=2)

In [27]:
import json

with open(r"C:\Users\ASUS\Hoc_DL\learning-DL\RAG_pdf\pipeline\luanngu.json", "r", encoding="utf-8") as f:
    dicc = json.load(f)

print(type(dicc))  # <class 'dict'>
# print(data)

<class 'dict'>


In [28]:
with open(r'C:\Users\ASUS\Hoc_DL\learning-DL\RAG_pdf\pipeline\test-1.txt', mode='r', encoding='utf-8') as f:
    thien = f.read()
    dicc['Lời mở đầu'] = thien
    
with open(r'C:\Users\ASUS\Hoc_DL\learning-DL\RAG_pdf\pipeline\test20.txt', mode='r', encoding='utf-8') as f:
    thien = f.read()
    dicc['Lời Kết'] = thien

In [29]:
dicc

{'Thiên 1': {'Tên thiên': 'Học nhi',
  'Chữ Hán': '学而',
  'Bài 1.1': '\n \n子曰: 学而时习之,不亦悅乎？ \n有朊自远方来,不亦乐乎? \n人不知 而不愠,不亦君子乎 ? \nTử viết: Học nhi thời tập chi, bất diệc duyệt hồ ? \nHữu bằng tự viễn phương lai, bất diệc lạc hồ ? \nNhân bất tri nhi bất uấn, bất diệc quân tử hồ ? \nKhổng tử nói: Học thì phải luyện tập, chẳng vui lắm sao ? \nCó bạn hữu nơi xa đến thăm, chẳng mừng lắm sao? \nNgười chẳng hiểu ta mà ta không buồn giận họ, thế chẳng phải người quân tử ư ? \n  \n(Lời bàn: Bài học đầu tiên, Khổng tử nói về niềm vui “học và hành”, niềm vui đón “bạn phương \nxa” và…nhắc đừng buồn khi có người hiểu lầm ta)  \n',
  'Bài 1.2': '  \n有子曰:“兲为人也孝悌, 而好犯上者,鲜矣; 不好犯上,而好作乱者,未之有也。君子务本,本立\n而道生。孝悌也者, 兲为仁之本与？” \nHữu tử viết: Kỳ vi nhân dã hiếu đễ, nhi hiếu phạm thượng giả, tiển hĩ; bất hiếu phạm thượng, nhi \nhiếu tác loạn giả, vị chi hữu dã. Quân tử vụ bản, bản lập nhi đạo sinh. Hiếu đễ dã giả, kỳ vi nhân \nchi bản dữ !\n \nHữu tử nói: Người biết hiếu thuận với cha mẹ, kính trọng người lớn tuổi hơ